In [1]:
!pip install genesis-world==0.3.10
!pip install tensordict

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


In [2]:
# ================================================================
# PHASE-4  DEPLOYMENT + EVALUATION  — COMPLETE UPDATED VERSION
# ================================================================
# Fixes:
#   1) PyTorch 2.6 torch.load(weights_only=True by default) issue
#   2) Missing create_environment() in previous script
#   3) Previous env.step() API mismatch for scalar deployment
#   4) Standalone evaluation of saved PPO checkpoint
#   5) Saves CSV outputs for report / comparison / visualization
# ================================================================

import os
import math
import json
import random
import inspect
import pickle
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.distributions import Normal
from tensordict import TensorDict
from tqdm import tqdm

# ================================================================
# CONFIG
# ================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

CHECKPOINT_PATH = "checkpoints/phase3/best.pth"
ARTIFACT_DIR = "artifacts/phase4"
EPISODES_PER_SCENARIO = 20

os.makedirs(ARTIFACT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ================================================================
# TRACK / ENV CONSTANTS
# ================================================================
OVAL_RADIUS  = 15.0
STRAIGHT_LEN = 40.0
TRACK_WIDTH  = 10.0
NUM_CONES    = 60
perimeter    = 2 * STRAIGHT_LEN + 2 * math.pi * OVAL_RADIUS

CONE_R      = 0.30
LANE_HALF   = TRACK_WIDTH / 2.0
CAR_HALF_W  = 1.25
LANE_SAFE   = float(LANE_HALF - (CAR_HALF_W + CONE_R + 0.25))
OFF_TRACK_THRESH = float(LANE_HALF - CAR_HALF_W - CONE_R)

OB_SIZE    = 1.5
MAX_STEER  = 0.55
BASE_SPEED = 8.0
MIN_SPEED  = 2.0
WHEELBASE  = 2.8

N_RAYS     = 41
FOV        = math.radians(170.0)
RAY_ANGLES = np.linspace(-FOV / 2, FOV / 2, N_RAYS).astype(np.float32)
LIDAR_MAX  = 30.0

NUM_OBS = 44
NUM_ACT = 1

# these will be overwritten from checkpoint metadata when available
MAX_EP_STEPS        = 420
COLLISION_THRESHOLD = 0.5
CRITICAL_THRESH     = 2.25

# ================================================================
# GLOBALS FILLED FROM CHECKPOINT
# ================================================================
SIM_DT   = None
OBS_MEAN = None
OBS_STD  = None
ACT_MEAN = 0.0
ACT_STD  = 1.0

# ================================================================
# HELPERS
# ================================================================
def wrap_pi(a):
    return (a + math.pi) % (2 * math.pi) - math.pi


def get_oval_pos(dist, offset=0.0):
    d = dist % perimeter
    if d < STRAIGHT_LEN:
        return (OVAL_RADIUS + offset, -STRAIGHT_LEN / 2 + d)
    d -= STRAIGHT_LEN
    if d < math.pi * OVAL_RADIUS:
        ang = (d / (math.pi * OVAL_RADIUS)) * math.pi
        R = OVAL_RADIUS + offset
        return (R * math.cos(ang), STRAIGHT_LEN / 2 + R * math.sin(ang))
    d -= math.pi * OVAL_RADIUS
    if d < STRAIGHT_LEN:
        return (-OVAL_RADIUS - offset, STRAIGHT_LEN / 2 - d)
    d -= STRAIGHT_LEN
    ang = math.pi + (d / (math.pi * OVAL_RADIUS)) * math.pi
    R = OVAL_RADIUS + offset
    return (R * math.cos(ang), -STRAIGHT_LEN / 2 + R * math.sin(ang))


def get_tangent_normal(dist, eps=0.25):
    x1, y1 = get_oval_pos(dist, 0.0)
    x2, y2 = get_oval_pos(dist + eps, 0.0)
    tx, ty = x2 - x1, y2 - y1
    nrm = math.sqrt(tx * tx + ty * ty) + 1e-8
    tx, ty = tx / nrm, ty / nrm
    return tx, ty, -ty, tx


def project_to_centerline(px, py, s_guess, window=18.0, samples=81):
    best_s, best_d2 = s_guess, 1e30
    s0 = s_guess - window
    ds = (2 * window) / (samples - 1)
    for i in range(samples):
        s = s0 + i * ds
        cx, cy = get_oval_pos(s, 0.0)
        d2 = (px - cx) ** 2 + (py - cy) ** 2
        if d2 < best_d2:
            best_d2, best_s = d2, s
    best_s = best_s % perimeter
    cx, cy = get_oval_pos(best_s, 0.0)
    tx, ty, nx, ny = get_tangent_normal(best_s)
    cte = (px - cx) * nx + (py - cy) * ny
    return best_s, cx, cy, tx, ty, nx, ny, float(cte)


def build_cone_xy():
    cone_xy = []
    for i in range(NUM_CONES):
        d = (i / NUM_CONES) * perimeter
        for sign in (-1, 1):
            ex, ey = get_oval_pos(d, offset=sign * LANE_HALF)
            cone_xy.append((float(ex), float(ey), float(CONE_R)))
    return np.array(cone_xy, dtype=np.float32)


CONE_XY = build_cone_xy()


def get_obstacle_configs():
    return [
        {"name": "Baseline",
         "dists": [0.12, 0.22, 0.34, 0.48, 0.65, 0.85],
         "lats":  [+1.6, -1.6, +2.0, -2.0, +1.5, -1.5]},
        {"name": "Wider",
         "dists": [0.10, 0.25, 0.40, 0.55, 0.70, 0.90],
         "lats":  [+2.0, -2.0, +1.8, -1.8, +2.0, -2.0]},
        {"name": "Dense",
         "dists": [0.08, 0.18, 0.28, 0.40, 0.52, 0.64, 0.76, 0.88],
         "lats":  [+1.5, -1.5, +1.8, -1.8, +1.5, -1.5, +1.8, -1.8]},
        {"name": "Offset",
         "dists": [0.15, 0.30, 0.45, 0.60, 0.75],
         "lats":  [+1.9, -1.7, +1.6, -1.9, +1.7]},
    ]


def build_obs_boxes(config):
    boxes = []
    for d_frac, lat in zip(config["dists"], config["lats"]):
        d = d_frac * perimeter
        tx, ty, nx, ny = get_tangent_normal(d)
        cx, cy = get_oval_pos(d, 0.0)
        ox, oy = cx + lat * nx, cy + lat * ny
        boxes.append((float(ox), float(oy), OB_SIZE / 2, OB_SIZE / 2))
    return np.array(boxes, dtype=np.float32)


SCENARIO_CONFIGS = get_obstacle_configs()
ALL_OBS_BOXES = {cfg["name"]: build_obs_boxes(cfg) for cfg in SCENARIO_CONFIGS}


def lidar_scan_fast(px, py, yaw, obs_boxes, max_range=30.0):
    if len(obs_boxes) == 0:
        return np.full(N_RAYS, max_range, dtype=np.float32)

    ga   = RAY_ANGLES + yaw
    dx   = np.sin(ga).astype(np.float32)
    dy   = np.cos(ga).astype(np.float32)
    P    = np.array([px, py], dtype=np.float32).reshape(1, 1, 2)
    D    = np.stack([dx, dy], axis=1).reshape(-1, 1, 2)
    B    = obs_boxes.reshape(1, -1, 4)
    Bmin = B[..., :2] - B[..., 2:]
    Bmax = B[..., :2] + B[..., 2:]

    invD    = 1.0 / (D + 1e-9)
    t0      = (Bmin - P) * invD
    t1      = (Bmax - P) * invD
    t_enter = np.max(np.minimum(t0, t1), axis=2)
    t_exit  = np.min(np.maximum(t0, t1), axis=2)
    hit     = (t_exit > t_enter) & (t_exit > 0)
    dists   = np.where(hit, np.maximum(0.0, t_enter), np.inf)
    return np.clip(np.min(dists, axis=1), 0.0, max_range).astype(np.float32)


def dist_point_to_aabb(px, py, cx, cy, hx, hy):
    return math.hypot(max(abs(px - cx) - hx, 0.0), max(abs(py - cy) - hy, 0.0))


def clearance_at_point(px, py, obs_boxes):
    best = 1e9
    for (cx, cy, hx, hy) in obs_boxes:
        best = min(best, dist_point_to_aabb(
            px, py, float(cx), float(cy), float(hx), float(hy)
        ))
    dx = CONE_XY[:, 0] - px
    dy = CONE_XY[:, 1] - py
    best = min(best, float(np.min(np.sqrt(dx * dx + dy * dy) - CONE_XY[:, 2])))
    return float(best)


def integrate_bicycle(px, py, yaw, v, delta, dt):
    yaw_rate = (v / WHEELBASE) * math.tan(delta)
    yaw  = wrap_pi(yaw + yaw_rate * dt)
    px  += v * math.sin(yaw) * dt
    py  += v * math.cos(yaw) * dt
    return px, py, yaw


# ================================================================
# NETWORK
# ================================================================
class BoundedGaussianMLP(nn.Module):
    is_recurrent = False

    def __init__(self,
                 input_dim=NUM_OBS,
                 output_dim=NUM_ACT,
                 hidden_dims=(256, 256),
                 activation="relu",
                 init_log_std=-1.2,
                 norm_lo=-1.0,
                 norm_hi=1.0):
        super().__init__()
        act_fn = {"elu": nn.ELU, "relu": nn.ReLU, "tanh": nn.Tanh}[activation]
        layers, d = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(d, h), act_fn()]
            d = h
        layers.append(nn.Linear(d, output_dim))
        self.net     = nn.Sequential(*layers)
        self.log_std = nn.Parameter(torch.full((output_dim,), init_log_std))
        self.norm_lo = float(norm_lo)
        self.norm_hi = float(norm_hi)
        self._dist   = None
        self._mean   = None
        self._std    = None

    def _x(self, obs):
        return obs["policy"] if isinstance(obs, TensorDict) else obs

    def _bounded_mean(self, x):
        z = self.net(x)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.clamp(z, self.norm_lo, self.norm_hi)

    def forward(self, obs, stochastic_output=False,
                masks=None, hidden_states=None, hidden_state=None):
        x    = self._x(obs)
        mean = self._bounded_mean(x)
        std  = torch.exp(torch.clamp(self.log_std, min=-4.0, max=1.0)).expand_as(mean)
        dist = Normal(mean, std)
        self._mean = mean
        self._std  = std
        self._dist = dist
        return dist.rsample() if stochastic_output else mean


# ================================================================
# CHECKPOINT LOAD
# ================================================================
def normalized_action_bounds(act_mean, act_std, max_steer):
    lo = (-max_steer - act_mean) / act_std
    hi = ( max_steer - act_mean) / act_std
    return float(lo), float(hi)


def load_checkpoint_robust(path):
    """
    PyTorch 2.6 changed torch.load default to weights_only=True.
    For your own trusted checkpoints, loading with weights_only=False is the
    simplest fix.
    """
    try:
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        return ckpt
    except TypeError:
        # for older PyTorch versions without weights_only arg
        ckpt = torch.load(path, map_location=DEVICE)
        return ckpt
    except pickle.UnpicklingError as e:
        raise RuntimeError(
            f"Failed to load checkpoint at {path}. "
            f"If this is your own trusted checkpoint, use weights_only=False."
        ) from e


def load_policy(path):
    global SIM_DT, OBS_MEAN, OBS_STD, ACT_MEAN, ACT_STD
    global MAX_EP_STEPS, COLLISION_THRESHOLD, CRITICAL_THRESH

    ckpt = load_checkpoint_robust(path)

    act_mean = float(ckpt["act_mean"][0])
    act_std  = float(ckpt["act_std"][0])

    norm_lo, norm_hi = normalized_action_bounds(act_mean, act_std, MAX_STEER)

    actor = BoundedGaussianMLP(
        input_dim=44,
        output_dim=1,
        hidden_dims=(256, 256),
        activation="relu",
        norm_lo=norm_lo,
        norm_hi=norm_hi,
    ).to(DEVICE)

    actor.load_state_dict(ckpt["actor"])
    actor.eval()

    obs_mean = torch.tensor(ckpt["obs_mean"], dtype=torch.float32, device=DEVICE)
    obs_std  = torch.tensor(ckpt["obs_std"], dtype=torch.float32, device=DEVICE)

    dt = float(ckpt["dt"])

    # read env metadata if available
    if "phase3_meta" in ckpt:
        meta = ckpt["phase3_meta"]
        MAX_EP_STEPS = int(meta.get("max_ep_steps", MAX_EP_STEPS))
        COLLISION_THRESHOLD = float(meta.get("collision_threshold", COLLISION_THRESHOLD))
        CRITICAL_THRESH = float(meta.get("critical_thresh", CRITICAL_THRESH))

    SIM_DT   = dt
    OBS_MEAN = obs_mean
    OBS_STD  = obs_std
    ACT_MEAN = act_mean
    ACT_STD  = act_std

    return actor, obs_mean, obs_std, act_mean, act_std, dt, ckpt


# ================================================================
# SINGLE DEPLOYMENT ENV
# ================================================================
class DeploymentEvalEnv:
    def __init__(self, scenario_name):
        assert scenario_name in ALL_OBS_BOXES, f"Unknown scenario: {scenario_name}"
        self.scenario_name = scenario_name
        self.obs_boxes = ALL_OBS_BOXES[scenario_name]

        self.px = 0.0
        self.py = 0.0
        self.yaw = 0.0
        self.speed = BASE_SPEED
        self.s_est = 0.0
        self.prev_steer = 0.0
        self.prev_yaw = 0.0
        self.ep_steps = 0
        self.ep_dist = 0.0
        self.min_clear = 1e9

    def reset(self):
        s_start = np.random.uniform(5.0, 13.0)
        cx0, cy0 = get_oval_pos(s_start, 0.0)
        tx, ty, nx, ny = get_tangent_normal(s_start)
        lat_noise = np.random.uniform(-0.2, 0.2)

        self.px = float(cx0) + lat_noise * nx
        self.py = float(cy0) + lat_noise * ny
        self.yaw = float(math.atan2(tx, ty)) + np.random.uniform(-0.025, 0.025)

        self.s_est = s_start
        self.speed = BASE_SPEED
        self.prev_steer = 0.0
        self.prev_yaw = self.yaw
        self.ep_steps = 0
        self.ep_dist = 0.0
        self.min_clear = 1e9

        return self._compute_obs()

    def _compute_obs(self):
        s_est, _, _, _, _, _, _, cte = project_to_centerline(
            self.px, self.py, self.s_est, window=18.0, samples=81
        )
        self.s_est = s_est

        ranges = lidar_scan_fast(self.px, self.py, self.yaw, self.obs_boxes, LIDAR_MAX)
        yaw_rate = (self.speed / WHEELBASE) * math.tan(float(self.prev_steer))

        obs = np.zeros((NUM_OBS,), dtype=np.float32)
        obs[:N_RAYS] = ranges
        obs[N_RAYS] = cte
        obs[N_RAYS + 1] = self.speed
        obs[N_RAYS + 2] = yaw_rate
        return obs

    def step(self, steer):
        steer = float(np.clip(steer, -MAX_STEER, MAX_STEER))
        s_old = self.s_est

        clearance = clearance_at_point(self.px, self.py, self.obs_boxes)
        speed = MIN_SPEED if clearance < CRITICAL_THRESH else BASE_SPEED
        self.speed = speed

        px_new, py_new, yaw_new = integrate_bicycle(
            self.px, self.py, self.yaw, speed, steer, SIM_DT
        )

        s_new, _, _, _, _, _, _, cte = project_to_centerline(
            px_new, py_new, s_old, window=18.0, samples=81
        )

        self.px, self.py, self.yaw = px_new, py_new, yaw_new
        self.s_est = s_new
        self.prev_yaw = yaw_new
        self.ep_steps += 1

        clearance = clearance_at_point(self.px, self.py, self.obs_boxes)
        self.min_clear = min(self.min_clear, clearance)

        off_track = abs(cte) > OFF_TRACK_THRESH

        ds = s_new - s_old
        if ds > perimeter / 2:
            ds -= perimeter
        elif ds < -perimeter / 2:
            ds += perimeter
        ds_fwd = max(0.0, float(ds))
        self.ep_dist += ds_fwd

        collision = clearance < COLLISION_THRESHOLD
        lap_done = self.ep_dist >= perimeter * 0.95
        timeout = self.ep_steps >= MAX_EP_STEPS
        done = collision or off_track or lap_done or timeout

        obs = self._compute_obs()

        info = {
            "progress": float(self.ep_dist),
            "clearance": float(clearance),
            "min_clearance": float(self.min_clear),
            "collision": bool(collision or off_track),
            "timeout": bool(timeout and not lap_done and not collision and not off_track),
            "lap_done": bool(lap_done),
            "off_track": bool(off_track),
            "cte": float(cte),
            "ds": float(ds_fwd),
            "steps": int(self.ep_steps),
            "scenario": self.scenario_name,
        }

        reward = 0.0  # evaluation only
        return obs, reward, done, info


def create_environment(scenario_name):
    return DeploymentEvalEnv(scenario_name)


# ================================================================
# EPISODE / EVAL
# ================================================================
def run_episode(actor, obs_mean, obs_std, act_mean, act_std, env):
    obs = env.reset()
    done = False

    total_progress = 0.0
    min_clearance = 999.0
    path_length = 0.0
    steps = 0
    last_info = None

    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        obs_t = (obs_t - obs_mean) / obs_std

        with torch.no_grad():
            act = actor(obs_t, stochastic_output=False)

        steer = float((act * act_std + act_mean).detach().cpu().numpy()[0][0])

        obs, reward, done, info = env.step(steer)

        total_progress = info["progress"]
        min_clearance = min(min_clearance, info["clearance"])
        path_length += info["ds"]
        steps += 1
        last_info = info

    return {
        "success": int(last_info["lap_done"]),
        "collision": int(last_info["collision"]),
        "timeout": int(last_info["timeout"]),
        "off_track": int(last_info["off_track"]),
        "progress_m": float(total_progress),
        "min_clearance_m": float(min_clearance),
        "path_length_m": float(path_length),
        "steps": int(steps),
        "final_cte": float(last_info["cte"]),
    }


def evaluate_policy(actor, obs_mean, obs_std, act_mean, act_std, env_builder, episodes_per_scenario=20):
    scenarios = ["Baseline", "Dense", "Offset", "Wider"]
    results = []

    for scenario in scenarios:
        env = env_builder(scenario)
        for ep in tqdm(range(episodes_per_scenario), desc=f"Evaluating {scenario}"):
            r = run_episode(actor, obs_mean, obs_std, act_mean, act_std, env)
            r["scenario"] = scenario
            r["episode"] = ep
            results.append(r)

    df = pd.DataFrame(results)

    summary = df.groupby("scenario").agg({
        "success": "mean",
        "collision": "mean",
        "timeout": "mean",
        "off_track": "mean",
        "progress_m": ["mean", "std", "max"],
        "min_clearance_m": ["mean", "std", "min"],
        "path_length_m": "mean",
        "steps": "mean",
        "final_cte": "mean",
    })

    summary.columns = [
        "_".join(col).strip("_") for col in summary.columns.to_flat_index()
    ]
    summary = summary.reset_index().rename(columns={
        "success_mean": "success_rate",
        "collision_mean": "collision_rate",
        "timeout_mean": "timeout_rate",
        "off_track_mean": "off_track_rate",
        "progress_m_mean": "mean_progress_m",
        "progress_m_std": "std_progress_m",
        "progress_m_max": "max_progress_m",
        "min_clearance_m_mean": "mean_min_clearance_m",
        "min_clearance_m_std": "std_min_clearance_m",
        "min_clearance_m_min": "worst_min_clearance_m",
        "path_length_m_mean": "mean_path_length_m",
        "steps_mean": "mean_steps",
        "final_cte_mean": "mean_final_cte",
    })

    return df, summary


# ================================================================
# OPTIONAL SCORE
# ================================================================
def compute_overall_metrics(df):
    overall = {
        "episodes_total": int(len(df)),
        "success_rate": float(df["success"].mean()),
        "collision_rate": float(df["collision"].mean()),
        "timeout_rate": float(df["timeout"].mean()),
        "off_track_rate": float(df["off_track"].mean()),
        "mean_progress_m": float(df["progress_m"].mean()),
        "std_progress_m": float(df["progress_m"].std()),
        "max_progress_m": float(df["progress_m"].max()),
        "mean_min_clearance_m": float(df["min_clearance_m"].mean()),
        "worst_min_clearance_m": float(df["min_clearance_m"].min()),
        "mean_path_length_m": float(df["path_length_m"].mean()),
        "mean_steps": float(df["steps"].mean()),
    }
    return overall


# ================================================================
# MAIN
# ================================================================
def main():
    print("Loading trained policy...")

    actor, obs_mean, obs_std, act_mean, act_std, dt, ckpt = load_policy(CHECKPOINT_PATH)

    print(f"Checkpoint loaded from: {CHECKPOINT_PATH}")
    print(f"Device: {DEVICE}")
    print(f"dt: {dt}")
    print(f"ACT_MEAN={act_mean:.6f}, ACT_STD={act_std:.6f}")
    print(f"MAX_EP_STEPS={MAX_EP_STEPS}, COLLISION_THRESHOLD={COLLISION_THRESHOLD}, CRITICAL_THRESH={CRITICAL_THRESH}")

    df, summary = evaluate_policy(
        actor=actor,
        obs_mean=obs_mean,
        obs_std=obs_std,
        act_mean=act_mean,
        act_std=act_std,
        env_builder=create_environment,
        episodes_per_scenario=EPISODES_PER_SCENARIO,
    )

    overall = compute_overall_metrics(df)

    episode_csv = os.path.join(ARTIFACT_DIR, "episode_results.csv")
    scenario_csv = os.path.join(ARTIFACT_DIR, "scenario_summary.csv")
    overall_csv = os.path.join(ARTIFACT_DIR, "overall_summary.csv")
    overall_json = os.path.join(ARTIFACT_DIR, "overall_summary.json")

    df.to_csv(episode_csv, index=False)
    summary.to_csv(scenario_csv, index=False)
    pd.DataFrame([overall]).to_csv(overall_csv, index=False)

    with open(overall_json, "w") as f:
        json.dump(overall, f, indent=2)

    print("\n================ EVALUATION SUMMARY ================\n")
    print(summary.to_string(index=False))

    print("\n================ OVERALL ================\n")
    for k, v in overall.items():
        print(f"{k}: {v}")

    print("\nSaved files:")
    print(f"  {episode_csv}")
    print(f"  {scenario_csv}")
    print(f"  {overall_csv}")
    print(f"  {overall_json}")


if __name__ == "__main__":
    main()

Loading trained policy...
Checkpoint loaded from: checkpoints/phase3/best.pth
Device: cuda
dt: 0.05
ACT_MEAN=-0.136163, ACT_STD=0.243136
MAX_EP_STEPS=420, COLLISION_THRESHOLD=0.5, CRITICAL_THRESH=2.25


Evaluating Wider: 100%|██████████| 20/20 [00:00<00:00, 76.75it/s]


================ EVALUATION SUMMARY ================

scenario  success_rate  collision_rate  timeout_rate  off_track_rate  mean_progress_m  std_progress_m  max_progress_m  mean_min_clearance_m  std_min_clearance_m  worst_min_clearance_m  mean_path_length_m  mean_steps  mean_final_cte
Baseline           0.0             1.0           0.0            1.00           5.2875        0.199918            5.40              1.422133             0.181570               1.229674              5.2875       23.55        3.493045
   Dense           0.0             1.0           0.0            0.60           4.9050        1.745287            8.55              0.876034             0.415355               0.417079              4.9050       31.85        2.332051
  Offset           0.0             1.0           0.0            1.00           4.9050        0.201246            5.40              1.469082             0.202272               1.186182              4.9050       22.25        3.494246
   Wider         